# NSMC 감성분류 — klue/bert-base 파인튜닝 프로젝트

한국어 NSMC(Naver Sentiment Movie Corpus)를 감성분류 태스크에 맞추고 `klue/bert-base`로 파인튜닝합니다.

## 환경 준비
---

In [1]:
!pip install -q transformers accelerate datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [2]:
import os
import gc
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline,
)
from evaluate import load as load_metric
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print("device:", device)

2.11.0+cu128
device: cuda


## STEP 1. `NSMC` 데이터 분석 및 Huggingface dataset 구성
---
앞서 MRPC 실습에서 배운 두 가지 방법(Huggingface datasets에서 불러오기 / 원본 텍스트 파일을 직접 가공하기)을 NSMC에도 그대로 적용합니다.

### 방법 1 — Huggingface datasets에서 불러오기

In [3]:
# [변경] load_dataset 대상을 glue/mrpc -> e9t/nsmc 로 변경.
# 이 데이터셋은 로딩 스크립트 방식이라 datasets 4 이상에서는 parquet 변환 브랜치를 지정해야 함
hf_nsmc_dataset = load_dataset("e9t/nsmc", revision="refs/convert/parquet")
print(hf_nsmc_dataset)
print(hf_nsmc_dataset['train'][0])
# 컬럼명이 위 출력과 다르게 나온다면, 아래 STEP들에서 'document'를 그 컬럼명으로 바꿔주세요.

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 11.1MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.71MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})
{'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}


### 방법 2 — 원본 텍스트 파일 직접 가공하기

NSMC 원본은 MRPC(Microsoft Research)와 달리 `id`, `document`, `label` 3개 컬럼으로만 이루어진 단순한 tab-구분 텍스트입니다.

In [4]:
!mkdir -p data
!wget -nc -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt -P data
!wget -nc -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt -P data
!wc -l data/ratings_*.txt

   50001 data/ratings_test.txt
  150001 data/ratings_train.txt
  200002 total


In [5]:
def parse_nsmc_file(file_path):
    """NSMC tab-구분 파일을 DataFrame으로 읽는 함수.
    MRPC 때는 컬럼 수가 들쭉날쭉해서 한 줄씩 직접 파싱했지만,
    NSMC는 컬럼이 항상 id/document/label 3개로 고정돼 있어 pandas로 바로 읽을 수 있다."""
    df = pd.read_csv(file_path, sep='\t')
    df = df.dropna(subset=['document'])           # 리뷰 본문이 비어있는(NaN) 행 제거
    df = df[df['document'].str.strip() != '']     # 공백만 있는 리뷰도 제거
    df['label'] = df['label'].astype(int)
    return df.reset_index(drop=True)


train_df = parse_nsmc_file('data/ratings_train.txt')
test_df = parse_nsmc_file('data/ratings_test.txt')

print("Train:", train_df.shape, " Test:", test_df.shape)
train_df.head()

Train: (149995, 3)  Test: (49997, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
# [변경] rename 대상이 MRPC(Quality/#1 String/#2 String) -> NSMC(필요 없음, 이미 document/label)로 바뀜.
# sentence1/sentence2 두 컬럼이던 MRPC와 달리, NSMC는 문장이 document 하나뿐이라 컬럼 구성이 더 단순함.
def to_hf_columns(df):
    return df[['document', 'label']].reset_index(drop=True)

train_df_hf = to_hf_columns(train_df)
test_df_hf = to_hf_columns(test_df)

# NSMC 원본에도 validation이 없으므로 train의 일부를 층화 분할로 떼어낸다.
# [변경] MRPC는 20%를 뗐지만, NSMC는 train 규모가 훨씬 커서(약 15만 건) 10%만 떼어도 검증 데이터 수가 충분함
train_split, val_split = train_test_split(
    train_df_hf, test_size=0.1, random_state=42, stratify=train_df_hf['label']
)

custom_nsmc_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_split, preserve_index=False),
    'validation': Dataset.from_pandas(val_split, preserve_index=False),
    'test': Dataset.from_pandas(test_df_hf, preserve_index=False),
})
print(custom_nsmc_dataset)
print(custom_nsmc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['document', 'label'],
        num_rows: 134995
    })
    validation: Dataset({
        features: ['document', 'label'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['document', 'label'],
        num_rows: 49997
    })
})
{'document': '이건 주인공들의 연기와 연출과 영상과 다 좋았는데 마지막회가서 너무 어거지로 후딱 끝내는것 같더라. 그게 아쉬워... 이야기가 조금더 길었으면 어땠을까 싶더라.', 'label': 1}


## STEP 2. klue/bert-base model 및 tokenizer 불러오기
---

In [7]:
# [변경] distilbert/distilbert-base-uncased(영어) -> klue/bert-base(한국어)
# NSMC가 한국어 리뷰 데이터이므로, 한국어 코퍼스로 사전학습된 klue/bert-base를 써야
# tokenizer의 vocab과 model이 한국어를 제대로 이해할 수 있다.
model_checkpoint = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
# num_labels=2 : NSMC는 긍정(1)/부정(0) 이진분류이므로 MRPC 때(paraphrase 여부)와 마찬가지로 2

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## STEP 3. 위에서 불러온 tokenizer으로 데이터셋을 전처리하고, model 학습 진행해 보기
---

NSMC는 리뷰 문장(`document`) 하나뿐이므로 넘기는 인자가 하나 입니다. 또한 klue/bert-base는 BERT 계열이라 DistilBERT와 달리 `token_type_ids`를 사용하므로 `return_token_type_ids=False`는 빼줍니다.

In [8]:
MAX_LENGTH = 128  # NSMC 리뷰는 대부분 짧은 문장이라 BERT 최대 길이(512)보다 훨씬 짧은 128로 충분

def transform(data):
    # [변경] sentence1, sentence2 두 개를 넘기던 MRPC와 달리 document 하나만 넘긴다
    return tokenizer(
        data['document'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    # return_token_type_ids=False를 빼서 BERT가 쓰는 token_type_ids를 그대로 생성하게 둔다

nsmc_dataset = custom_nsmc_dataset.map(transform, batched=True)

nsmc_train_dataset = nsmc_dataset['train']
nsmc_val_dataset = nsmc_dataset['validation']
nsmc_test_dataset = nsmc_dataset['test']
print(nsmc_train_dataset)

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

Dataset({
    features: ['document', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 134995
})


NSMC는 GLUE 태스크가 아니므로 `evaluate.load('glue', 'mrpc')` 대신 accuracy 지표를 직접 계산합니다.

In [9]:
accuracy_metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

## STEP 4. Fine-tuning을 통하여 모델 성능(accuracy) 향상시키기
---
우선 1 epoch로 목표 달성 여부를 확인하고, 부족할 경우 epoch를 늘릴 것입니다. 'warmup_ratio'로는 오류가 나서 `warmup_steps`를 사용했습니다. colab v5에서는 ratio가 없어지고 steps로 통합되었다고 합니다.

In [10]:
output_dir_tuned = './results/nsmc_tuned'

training_args_tuned = TrainingArguments(
    output_dir_tuned,
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,   # [변경] 8(MRPC 강의안 기준) -> 64. NSMC는 13만 건이 넘어 MRPC(수천 건)보다 훨씬 크므로,
                                       # 한 배치에 더 많은 데이터를 묶어 GPU 병렬 연산을 최대한 활용해야 학습 시간이 감당할 수 있는 수준이 된다.
    per_device_eval_batch_size=128,   # [변경] 평가는 역전파(gradient 계산)가 없어 학습보다 메모리를 덜 쓰므로, 학습 batch보다 크게 잡아도 된다.
    num_train_epochs=1,               # [변경] 3 -> 1. 데이터 규모 자체가 크기 때문에 1 epoch만으로도 학습량이 충분한 경우가 많다.
                                       # 목표(90%) 달성 여부를 먼저 확인하고, 부족하면 바로 아래 안내대로 늘린다.
    warmup_steps=0.1,                 # [변경] transformers v5부터 warmup_ratio가 폐지되고 warmup_steps로 통합됨.
                                       # warmup_steps에 1 이상 정수를 넣으면 "스텝 수", 1 미만 소수를 넣으면 "비율"로 해석되므로
                                       # 기존 warmup_ratio=0.1과 동일한 의미
    weight_decay=0.01,
    logging_steps=200,
    report_to="none",
)

trainer_tuned = Trainer(
    model=model,
    args=training_args_tuned,
    train_dataset=nsmc_train_dataset,
    eval_dataset=nsmc_val_dataset,
    compute_metrics=compute_metrics,
)
train_result_tuned = trainer_tuned.train()

step4_eval = trainer_tuned.evaluate()
print("STEP4 validation 결과:", step4_eval)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.238810,0.238327,0.904000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.238810,0.238327,1,0.904000


STEP4 validation 결과: {'eval_loss': 0.2383267730474472, 'eval_accuracy': 0.904}


In [11]:
step4_test_eval = trainer_tuned.evaluate(nsmc_test_dataset)
print("STEP4 test 결과:", step4_test_eval)

Training Loss,Validation Loss,Epoch,Accuracy
0.238810,0.239100,1,0.902134


STEP4 test 결과: {'eval_loss': 0.23910024762153625, 'eval_accuracy': 0.9021341280476829}


In [12]:
del trainer_tuned, model
gc.collect()
torch.cuda.empty_cache()

## STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교
---
STEP 3~4에서는 `padding='max_length'`로 모든 문장을 128 토큰까지 고정 padding했습니다. 이번엔 동적 패딩(dynamic padding, `DataCollatorWithPadding`)과 버켓팅(bucketing, 비슷한 길이의 문장끼리 배치로 묶기)을 적용해 STEP 4와 훈련 시간·성능을 비교합니다.

In [13]:
def transform_dynamic(data):
    # [변경] padding='max_length' 제거 — 배치 단위로 DataCollatorWithPadding이 동적으로 채우도록 맡긴다
    return tokenizer(data['document'], truncation=True, max_length=MAX_LENGTH)

nsmc_dataset_dynamic = custom_nsmc_dataset.map(transform_dynamic, batched=True)

nsmc_train_dynamic = nsmc_dataset_dynamic['train']
nsmc_val_dynamic = nsmc_dataset_dynamic['validation']
nsmc_test_dynamic = nsmc_dataset_dynamic['test']

Map:   0%|          | 0/134995 [00:00<?, ? examples/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# 배치를 만들 때마다, 그 배치 안에서 가장 긴 문장 기준으로만 padding을 채워주는 역할 (동적 패딩)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

output_dir_bucket = './results/nsmc_bucket'

training_args_bucket = TrainingArguments(
    output_dir_bucket,
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=64,   # [변경] STEP 4와 동일하게 맞춤 — padding 방식 외 조건을 똑같이 둬야 비교가 공정함
    per_device_eval_batch_size=128,   # [변경] STEP 4와 동일
    num_train_epochs=1,               # [변경] STEP 4와 동일 (STEP 4에서 epoch를 늘렸다면 여기도 같이 늘릴 것)
    warmup_steps=0.1,                 # [변경] transformers v5부터 warmup_ratio가 폐지되고 warmup_steps로 통합됨 (STEP 4와 동일한 이유)
    weight_decay=0.01,
    logging_steps=200,
    report_to="none",
    train_sampling_strategy="group_by_length",
    # [변경] 추가 — transformers v5부터 group_by_length=True 대신 쓰는 설정(이 STEP 안내문에 적힌 표기 그대로).
    # 비슷한 길이의 문장끼리 같은 배치로 묶어서, 배치마다 낭비되는 padding 토큰 수를 줄인다.
)

trainer_bucket = Trainer(
    model=model,
    args=training_args_bucket,
    train_dataset=nsmc_train_dynamic,
    eval_dataset=nsmc_val_dynamic,
    data_collator=data_collator,   # [변경] 추가 — 동적 패딩을 실제로 수행하는 부분
    compute_metrics=compute_metrics,
)
train_result_bucket = trainer_bucket.train()

step5_eval = trainer_bucket.evaluate()
print("STEP5 validation 결과:", step5_eval)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.246331,0.238140,0.902933


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.246331,0.238140,1,0.902933


STEP5 validation 결과: {'eval_loss': 0.23813988268375397, 'eval_accuracy': 0.9029333333333334}


In [15]:
step5_test_eval = trainer_bucket.evaluate(nsmc_test_dynamic)
print("STEP5 test 결과:", step5_test_eval)

Training Loss,Validation Loss,Epoch,Accuracy
0.246331,0.240554,1,0.900294


STEP5 test 결과: {'eval_loss': 0.24055412411689758, 'eval_accuracy': 0.9002940176410584}


### STEP 4 vs STEP 5 비교

두 방식의 `train_runtime`(학습 소요 시간)과 accuracy를 나란히 확인합니다.

In [16]:
print(f"{'':20s}{'STEP4 (고정 padding)':>22s}{'STEP5 (bucketing)':>22s}")
print(f"{'val accuracy':20s}{step4_eval['eval_accuracy']:>22.4f}{step5_eval['eval_accuracy']:>22.4f}")
print(f"{'test accuracy':20s}{step4_test_eval['eval_accuracy']:>22.4f}{step5_test_eval['eval_accuracy']:>22.4f}")
print(f"{'train_runtime(초)':20s}{train_result_tuned.metrics['train_runtime']:>22.1f}{train_result_bucket.metrics['train_runtime']:>22.1f}")

                        STEP4 (고정 padding)     STEP5 (bucketing)
val accuracy                        0.9040                0.9029
test accuracy                       0.9021                0.9003
train_runtime(초)                    2971.4                 930.1


## 결과 확인
학습이 끝난 모델(STEP 5, bucketing 적용)에 새 리뷰 문장을 넣어 정상적으로 감성을 분류하는지 확인합니다.

In [17]:
# model.config에 라벨 이름을 지정해두면 pipeline이 숫자(0/1) 대신 이 이름으로 결과를 보여준다.
# (1강에서 배운 "model.config로 속성에 접근/지정할 수 있다"는 내용을 그대로 활용)
model.config.id2label = {0: "부정", 1: "긍정"}
model.config.label2id = {"부정": 0, "긍정": 1}

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

examples = [
    "이 영화 진짜 재밌었어요 다시 보고 싶다",
    "시간 낭비였다 최악의 영화",
    "배우 연기는 좋았지만 스토리가 아쉬웠어요",
    "역대급 명작입니다 강추",
]

results = sentiment_classifier(examples)
for sentence, result in zip(examples, results):
    print(sentence, '->', result['label'], f"(score={result['score']:.4f})")

이 영화 진짜 재밌었어요 다시 보고 싶다 -> 긍정 (score=0.9881)
시간 낭비였다 최악의 영화 -> 부정 (score=0.9985)
배우 연기는 좋았지만 스토리가 아쉬웠어요 -> 부정 (score=0.9498)
역대급 명작입니다 강추 -> 긍정 (score=0.9819)


## 회고
---

- **최종 accuracy (validation / test)**:

   고정 padding: 0.9040 / 0.9021   
  bucketing: 0.9029 / 0.9003

- **STEP 4(고정 padding) vs STEP 5(bucketing) 비교**:

  성능은 거의 비슷했다. 고정 padding이 0.002정도 정확도가 높았다.
  
  반면에 연산속도는 bucketig 쪽이 3배 정도 빨랐다.(2971초 vs 930초)
- **MRPC 코드를 NSMC로 바꾸면서 가장 힘들었던/인상적이었던 부분**:

- **아쉬웠던 점 / 다음에 시도해보고 싶은 것**:
